# Trader Type Prediction Notebook

This notebook loads the trained model from `models/`, reads `layering.csv` and `spoofing.csv`, builds model features, and predicts the `trader_type` target.

In [6]:
from pathlib import Path

import pickle

import subprocess

import sys



import numpy as np

import pandas as pd



PROJECT_FOLDER_NAME = 'models-20260313T125452Z-1-001'

# Set this if you know the exact Drive path, for example:

# MANUAL_BASE_DIR = '/content/drive/MyDrive/new/models-20260313T125452Z-1-001'

MANUAL_BASE_DIR = None





def mount_google_drive():

    try:

        from google.colab import drive

    except ImportError:

        return False



    drive.mount('/content/drive', force_remount=False)

    return True





def contains_required_files(base_dir: Path) -> bool:

    required_paths = [

        base_dir / 'layering.csv',

        base_dir / 'spoofing.csv',

        base_dir / 'models' / 'feature_cols.pkl',

        base_dir / 'models' / 'manipulation_detector.pkl',

    ]

    return all(path.exists() for path in required_paths)





def candidate_roots():

    roots = [

        Path.cwd(),

        Path.cwd().parent,

        Path('/content'),

        Path('/content/drive'),

        Path('/content/drive/MyDrive'),

        Path('/content/drive/Shareddrives'),

    ]

    seen = set()

    ordered_roots = []

    for root in roots:

        root_str = str(root)

        if root_str not in seen and root.exists():

            seen.add(root_str)

            ordered_roots.append(root)

    return ordered_roots





def resolve_base_dir(project_folder_name: str, manual_base_dir: str | None = None) -> Path:

    if manual_base_dir:

        manual_path = Path(manual_base_dir)

        if contains_required_files(manual_path):

            return manual_path

        raise FileNotFoundError(f'MANUAL_BASE_DIR does not contain the required files: {manual_path}')



    direct_candidates = [

        Path.cwd(),

        Path.cwd() / project_folder_name,

        Path.cwd().parent / project_folder_name,

    ]



    for root in candidate_roots():

        direct_candidates.extend([

            root / project_folder_name,

            root / 'new' / project_folder_name,

            root / 'My Drive' / project_folder_name,

            root / 'new',

            root / 'My Drive' / 'new' / project_folder_name,

        ])



    seen = set()

    for candidate in direct_candidates:

        candidate_str = str(candidate)

        if candidate_str in seen:

            continue

        seen.add(candidate_str)

        if contains_required_files(candidate):

            return candidate



    for root in candidate_roots():

        folder_matches = list(root.rglob(project_folder_name))

        for match in folder_matches:

            if match.is_dir() and contains_required_files(match):

                return match



        model_matches = list(root.rglob('feature_cols.pkl'))

        for model_match in model_matches:

            maybe_base = model_match.parent.parent

            if contains_required_files(maybe_base):

                return maybe_base



    searched_roots = ', '.join(str(root) for root in candidate_roots())

    raise FileNotFoundError(

        'Could not find the uploaded project folder. Checked roots: '

        f"{searched_roots}. If needed, set MANUAL_BASE_DIR to the exact folder containing layering.csv, spoofing.csv, and models/."

    )





drive_mounted = mount_google_drive()

BASE_DIR = resolve_base_dir(PROJECT_FOLDER_NAME, MANUAL_BASE_DIR)

MODELS_DIR = BASE_DIR / 'models'

LAYERING_CSV = BASE_DIR / 'layering.csv'

SPOOFING_CSV = BASE_DIR / 'spoofing.csv'



print('Drive mounted:', drive_mounted)

print('Base directory:', BASE_DIR)

print('Models directory:', MODELS_DIR)

print('Layering CSV exists:', LAYERING_CSV.exists())

print('Spoofing CSV exists:', SPOOFING_CSV.exists())

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Drive mounted: True
Base directory: /content/drive/MyDrive/manipulation_data
Models directory: /content/drive/MyDrive/manipulation_data/models
Layering CSV exists: True
Spoofing CSV exists: True


In [7]:
# The serialized model depends on xgboost for loading.
try:
    import xgboost  # noqa: F401
except ImportError:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'xgboost'])

with open(MODELS_DIR / 'feature_cols.pkl', 'rb') as f:
    feature_cols = pickle.load(f)

with open(MODELS_DIR / 'manipulation_detector.pkl', 'rb') as f:
    model = pickle.load(f)

print('Number of expected features:', len(feature_cols))
print('First 10 feature names:', feature_cols[:10])
print('Loaded model type:', type(model).__name__)

Number of expected features: 23
First 10 feature names: ['cancel_rate', 'zero_fill_cancel_rate', 'cancel_to_order_ratio', 'avg_fill_ratio', 'min_fill_ratio', 'avg_time_to_cancel', 'min_time_to_cancel', 'std_time_to_cancel', 'limit_order_ratio', 'market_order_ratio']
Loaded model type: XGBClassifier


In [8]:
layering_df = pd.read_csv(LAYERING_CSV)
spoofing_df = pd.read_csv(SPOOFING_CSV)
raw_df = pd.concat([layering_df, spoofing_df], ignore_index=True)

print('Layering shape:', layering_df.shape)
print('Spoofing shape:', spoofing_df.shape)
print('Combined shape:', raw_df.shape)
print('Columns:', list(raw_df.columns))

Layering shape: (240343, 25)
Spoofing shape: (236473, 25)
Combined shape: (476816, 25)
Columns: ['order_id', 'instrument_id', 'order_type', 'side', 'order_status_event', 'user_id', 'trade_id', 'buyer_user_id', 'seller_user_id', 'aggressor_side', 'market_phase', 'device_id_hash', 'trader_type', 'buyer_trader_type', 'seller_trader_type', 'price', 'quantity', 'filled_quantity', 'remaining_quantity', 'is_short_sell', 'order_submit_timestamp', 'order_cancel_timestamp', 'match_engine_timestamp', 'timestamp', 'tradertype']


In [9]:
def to_numeric(series, default=0.0):
    return pd.to_numeric(series, errors='coerce').fillna(default)


def to_datetime_safe(series):
    return pd.to_datetime(series, errors='coerce', utc=True)


def mode_or_unknown(series):
    s = series.dropna().astype(str)
    if s.empty:
        return 'unknown'
    return s.mode().iloc[0]


def build_user_features(df):
    work = df.copy()

    for col in ['quantity', 'filled_quantity', 'remaining_quantity', 'price']:
        if col not in work.columns:
            work[col] = 0
        work[col] = to_numeric(work[col], 0.0)

    for col in ['order_type', 'side', 'order_status_event', 'trade_id', 'instrument_id', 'buyer_user_id', 'seller_user_id', 'trader_type']:
        if col not in work.columns:
            work[col] = np.nan

    work['order_submit_dt'] = to_datetime_safe(work.get('order_submit_timestamp'))
    work['order_cancel_dt'] = to_datetime_safe(work.get('order_cancel_timestamp'))
    work['match_engine_dt'] = to_datetime_safe(work.get('match_engine_timestamp'))

    status = work['order_status_event'].astype(str).str.lower()
    order_type = work['order_type'].astype(str).str.lower()
    side = work['side'].astype(str).str.lower()

    work['is_cancel'] = status.str.contains('cancel', na=False)
    work['is_limit'] = order_type.eq('limit')
    work['is_market'] = order_type.eq('market')
    work['is_buy'] = side.eq('buy')
    work['is_sell'] = side.eq('sell')
    work['has_trade'] = work['trade_id'].notna()

    qty = work['quantity'].replace(0, np.nan)
    work['fill_ratio'] = (work['filled_quantity'] / qty).replace([np.inf, -np.inf], np.nan).fillna(0.0).clip(0, 1)

    work['time_to_cancel'] = (work['order_cancel_dt'] - work['order_submit_dt']).dt.total_seconds()
    work['time_to_cancel'] = work['time_to_cancel'].replace([np.inf, -np.inf], np.nan)

    work['engine_latency'] = (work['match_engine_dt'] - work['order_submit_dt']).dt.total_seconds()
    work['engine_latency'] = work['engine_latency'].replace([np.inf, -np.inf], np.nan)

    # Relative deviation from instrument median price.
    inst_med = work.groupby('instrument_id', dropna=False)['price'].transform('median')
    denom = inst_med.replace(0, np.nan)
    work['price_deviation'] = ((work['price'] - inst_med).abs() / denom).replace([np.inf, -np.inf], np.nan).fillna(0.0)

    grp = work.groupby('user_id', dropna=False)

    out = pd.DataFrame(index=grp.size().index)
    out['cancel_rate'] = grp['is_cancel'].mean()
    out['zero_fill_cancel_rate'] = grp.apply(lambda g: ((g['is_cancel']) & (g['filled_quantity'] <= 0)).mean())
    out['cancel_to_order_ratio'] = grp['is_cancel'].sum() / grp.size()
    out['avg_fill_ratio'] = grp['fill_ratio'].mean()
    out['min_fill_ratio'] = grp['fill_ratio'].min()
    out['avg_time_to_cancel'] = grp['time_to_cancel'].mean()
    out['min_time_to_cancel'] = grp['time_to_cancel'].min()
    out['std_time_to_cancel'] = grp['time_to_cancel'].std()
    out['limit_order_ratio'] = grp['is_limit'].mean()
    out['market_order_ratio'] = grp['is_market'].mean()

    self_trade = (work['buyer_user_id'].astype(str) == work['seller_user_id'].astype(str)) & work['buyer_user_id'].notna()
    out['self_trade_count'] = work[self_trade].groupby('user_id', dropna=False).size().reindex(out.index, fill_value=0)

    out['both_sides_manip'] = grp['side'].nunique(dropna=True).gt(1).astype(int)
    out['avg_engine_latency'] = grp['engine_latency'].mean()
    out['max_engine_latency'] = grp['engine_latency'].max()
    out['unique_sides'] = grp['side'].nunique(dropna=True)
    out['unique_instruments'] = grp['instrument_id'].nunique(dropna=True)

    buy_counts = grp['is_buy'].sum()
    sell_counts = grp['is_sell'].sum()
    total_side = (buy_counts + sell_counts).replace(0, np.nan)
    out['order_book_imbalance'] = ((buy_counts - sell_counts).abs() / total_side).fillna(0.0)

    def cancel_regularity(g):
        ts = g.loc[g['is_cancel'], 'order_cancel_dt'].dropna().sort_values()
        if ts.shape[0] < 3:
            return 0.0
        return ts.diff().dt.total_seconds().std() or 0.0

    out['cancel_timing_regularity'] = grp.apply(cancel_regularity)
    out['avg_price_deviation'] = grp['price_deviation'].mean()

    trades_per_user = grp['has_trade'].sum()
    orders_per_user = grp.size()
    out['quote_to_trade_ratio'] = (orders_per_user / trades_per_user.replace(0, np.nan)).replace([np.inf, -np.inf], np.nan).fillna(0.0)

    out['latency_cancel_interaction'] = out['avg_engine_latency'].fillna(0.0) * out['cancel_rate'].fillna(0.0)

    def cancel_intensity(g):
        t0 = g['order_submit_dt'].min()
        t1 = g['order_submit_dt'].max()
        if pd.isna(t0) or pd.isna(t1) or t0 == t1:
            return float(g['is_cancel'].sum())
        span = max((t1 - t0).total_seconds(), 1.0)
        return float(g['is_cancel'].sum()) / span

    out['cancel_intensity_ratio'] = grp.apply(cancel_intensity)
    out['fill_cancel_interaction'] = out['avg_fill_ratio'].fillna(0.0) * out['cancel_rate'].fillna(0.0)

    out['trader_type'] = grp['trader_type'].apply(mode_or_unknown)

    out = out.replace([np.inf, -np.inf], np.nan).fillna(0.0)
    out['trader_type'] = grp['trader_type'].apply(mode_or_unknown)

    return out.reset_index().rename(columns={'index': 'user_id'})

In [10]:
user_features = build_user_features(raw_df)

for col in feature_cols:
    if col not in user_features.columns:
        user_features[col] = 0.0

X = user_features[feature_cols].copy()
X = X.replace([np.inf, -np.inf], np.nan).fillna(0.0)

user_features['predicted_trader_type'] = model.predict(X)

preview_cols = ['user_id', 'trader_type', 'predicted_trader_type']
print(user_features[preview_cols].head(20))

if 'trader_type' in user_features.columns:
    accuracy = (user_features['trader_type'].astype(str) == user_features['predicted_trader_type'].astype(str)).mean()
    print(f'Approx user-level accuracy against CSV trader_type: {accuracy:.4f}')

/tmp/ipykernel_364/2862923594.py:61: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  out['zero_fill_cancel_rate'] = grp.apply(lambda g: ((g['is_cancel']) & (g['filled_quantity'] <= 0)).mean())
/tmp/ipykernel_364/2862923594.py:91: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  out['cancel_timing_regularity'] = grp.apply(cancel_regularity)
/tmp/ipykernel_364/2862923594.py:108: DeprecationWarning: DataFrameGroupB

    user_id  trader_type  predicted_trader_type
0         1  MANIPULATOR                      1
1         2  MANIPULATOR                      1
2         3  MANIPULATOR                      1
3         4  MANIPULATOR                      1
4         5  MANIPULATOR                      1
5         6  MANIPULATOR                      1
6         7  MANIPULATOR                      1
7         8  MANIPULATOR                      1
8         9  MANIPULATOR                      1
9        10  MANIPULATOR                      1
10       11  MANIPULATOR                      1
11       12  MANIPULATOR                      1
12       13  MANIPULATOR                      1
13       14  MANIPULATOR                      1
14       15  MANIPULATOR                      1
15       16       NORMAL                      0
16       17       NORMAL                      0
17       18       NORMAL                      0
18       19       NORMAL                      0
19       20       NORMAL                

In [11]:
output_path = BASE_DIR / 'trader_type_predictions.csv'
user_features.to_csv(output_path, index=False)
print('Saved predictions to:', output_path)
print('Rows saved:', len(user_features))

Saved predictions to: /content/drive/MyDrive/manipulation_data/trader_type_predictions.csv
Rows saved: 3015
